In [1]:
# Celda X1 - Configuracion, carga y panel de precios por ticker-dia (2020-2024, CRSP)
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/Users/ppizam/Claude/Master Thesis')
MATRIX = BASE / 'Desarrollo' / 'Metodologia' / 'Matrix'
EV = MATRIX / 'eventos'
LM = BASE / 'Desarrollo' / 'Metodologia' / 'Lista Maestra de Tickers' / 'Lista maestra V2'

cat = pd.read_csv(EV / 'eventos_atencion_v2_principal_final.csv',
                  keep_default_na=False, na_values=[''])
cat['fecha_inicio'] = pd.to_datetime(cat['fecha_inicio'])
cat['fecha_pico'] = pd.to_datetime(cat['fecha_pico'])
cat['fecha_fin'] = pd.to_datetime(cat['fecha_fin'])
print(f'catalogo: {len(cat)} eventos, {cat.ticker.nunique()} tickers')

padron = pd.read_csv(LM / 'padron_vigencias_2020_2026_ver03_2.csv',
                     keep_default_na=False, na_values=[''])
padron['fecha_inicio'] = pd.to_datetime(padron['fecha_inicio'])
padron['fecha_fin'] = pd.to_datetime(padron['fecha_fin'])

px = pd.read_csv(EV / 'precios_dsf_eventos_2020_2024.csv',
                 keep_default_na=False, na_values=[''])
px['date'] = pd.to_datetime(px['date'])
print(f'precios CRSP: {len(px):,} renglones, {px.permno.nunique()} permnos')

# mapear cada (permno, fecha) al ticker vigente segun las vidas del padron
vidas = padron[padron['ticker'].isin(set(cat.ticker)) & padron['permno'].notna()].copy()
vidas['permno'] = vidas['permno'].astype(int)

piezas = []
por_permno = dict(tuple(px.groupby('permno')))
for _, v in vidas.iterrows():
    bloque = por_permno.get(int(v['permno']))
    if bloque is None:
        continue
    tramo = bloque[(bloque.date >= v['fecha_inicio']) & (bloque.date <= v['fecha_fin'])]
    if len(tramo):
        piezas.append(tramo.assign(ticker=v['ticker']))

panel = pd.concat(piezas, ignore_index=True)
panel = panel.drop_duplicates(['ticker', 'date']).sort_values(['ticker', 'date'])
panel = panel[['ticker', 'date', 'prc', 'ret', 'vol', 'shrout']]
panel.to_csv(EV / 'panel_precios_2020_2024.csv', index=False)
print(f'panel ticker-dia: {len(panel):,} renglones, {panel.ticker.nunique()} tickers')
print(panel[panel.ticker == 'GME'].tail(3).to_string(index=False))

catalogo: 2791 eventos, 1012 tickers
precios CRSP: 1,025,942 renglones, 977 permnos
panel ticker-dia: 935,089 renglones, 987 tickers
ticker       date   prc       ret        vol   shrout
   GME 2024-12-27 32.20 -0.023947 10141662.0 446510.0
   GME 2024-12-30 32.01 -0.005901  9593140.0 446510.0
   GME 2024-12-31 31.34 -0.020931  7395297.0 446800.0


In [2]:
# Celda X2 - Cola de precios 2025-jun2026 via Yahoo para tickers con eventos en ese periodo
import yfinance as yf
import time

necesitan = sorted(set(cat[cat.fecha_fin >= '2025-01-01'].ticker))
print(f'tickers con eventos en 2025-2026: {len(necesitan)}')

destino = EV / 'precios_yahoo_2025_2026.csv'
hechos = set()
if destino.exists():
    hechos = set(pd.read_csv(destino, keep_default_na=False, na_values=[''])['ticker'])
    print(f'checkpoint: {len(hechos)} tickers ya descargados')

nuevos = []
errores = []
for i, t in enumerate(necesitan):
    if t in hechos:
        continue
    try:
        h = yf.download(t, start='2024-11-01', end='2026-07-01',
                        auto_adjust=False, progress=False)
        if len(h) == 0:
            errores.append(t)
            continue
        h = h.reset_index()
        if isinstance(h.columns, pd.MultiIndex):
            h.columns = [c[0] for c in h.columns]
        nuevos.append(pd.DataFrame({
            'ticker': t,
            'date': pd.to_datetime(h['Date']),
            'close': h['Close'].values,
            'volume': h['Volume'].values,
        }))
    except Exception as e:
        errores.append(t)
    time.sleep(0.4)
    if (i + 1) % 25 == 0:
        print(f'{i + 1}/{len(necesitan)}', flush=True)
        if nuevos:
            parcial = pd.concat(nuevos, ignore_index=True)
            if destino.exists():
                parcial = pd.concat([pd.read_csv(destino, keep_default_na=False,
                                     na_values=[''], parse_dates=['date']), parcial])
            parcial.to_csv(destino, index=False)
            nuevos = []

if nuevos:
    parcial = pd.concat(nuevos, ignore_index=True)
    if destino.exists():
        parcial = pd.concat([pd.read_csv(destino, keep_default_na=False,
                             na_values=[''], parse_dates=['date']), parcial])
    parcial.to_csv(destino, index=False)

final = pd.read_csv(destino, keep_default_na=False, na_values=[''])
print(f'\ncola Yahoo: {len(final):,} renglones, {final.ticker.nunique()} tickers')
print(f'sin datos en Yahoo ({len(errores)}): {errores[:20]}')

tickers con eventos en 2025-2026: 250
25/250


$BRK: possibly delisted; no price data found  (1d 2024-11-01 -> 2026-07-01)

1 Failed download:
['BRK']: possibly delisted; no price data found  (1d 2024-11-01 -> 2026-07-01)


50/250


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DAWN"}}}
$DAWN: possibly delisted; no timezone found

1 Failed download:
['DAWN']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DAY"}}}
$DAY: possibly delisted; no timezone found

1 Failed download:
['DAY']: possibly delisted; no timezone found


75/250


...

100/250
125/250
150/250
175/250


$PHX: possibly delisted; no timezone found

1 Failed download:
['PHX']: possibly delisted; no timezone found


200/250


$SHOT: possibly delisted; no price data found  (1d 2024-11-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1730433600, endDate = 1782878400")

1 Failed download:
['SHOT']: possibly delisted; no price data found  (1d 2024-11-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1730433600, endDate = 1782878400")


225/250
250/250

cola Yahoo: 97,169 renglones, 244 tickers
sin datos en Yahoo (6): ['BRK', 'DAWN', 'DAY', 'HTBK', 'PHX', 'SHOT']


In [3]:
# Celda X3 v2 - Panel unico de precios 2020-jun2026 (CRSP + dsf_v2 2025 + cola Yahoo 2026)
# Migracion 12-ago-2026: el 2025 ahora sale de crsp.dsf_v2 (formato CIZ, publicado
# hasta 2025-12-31): retornos con dividendos y sin artefactos de splits de Yahoo.
# Yahoo queda como respaldo 2025 (solo tickers fuera del universo CRSP) y como
# fuente de ene-jun 2026 hasta que CRSP publique 2026.
crsp = pd.read_csv(EV / 'panel_precios_2020_2024.csv',
                   keep_default_na=False, na_values=[''], parse_dates=['date'])
crsp = crsp.rename(columns={'prc': 'close', 'vol': 'volume'})
crsp['fuente'] = 'crsp'

# --- bloque 2025 desde dsf_v2, mapeado a ticker por las vidas del padron 03.2 ---
px25 = pd.read_csv(LM / 'dsf_v2_precios_2025.csv', parse_dates=['dlycaldt'])
vidas25 = padron[padron['ticker'].isin(set(cat.ticker)) & padron['permno'].notna()].copy()
vidas25['permno'] = vidas25['permno'].astype(float).astype(int)
por_permno = dict(tuple(px25.groupby('permno')))
piezas = []
for _, v in vidas25.iterrows():
    b = por_permno.get(v['permno'])
    if b is None:
        continue
    fin = v['fecha_fin'] if pd.notna(v['fecha_fin']) else pd.Timestamp('2026-06-30')
    tramo = b[(b.dlycaldt >= v['fecha_inicio']) & (b.dlycaldt <= fin)]
    if len(tramo):
        piezas.append(tramo.assign(ticker=v['ticker']))
v2 = pd.concat(piezas, ignore_index=True).drop_duplicates(['ticker', 'dlycaldt'])
v2 = pd.DataFrame({'ticker': v2['ticker'], 'date': v2['dlycaldt'],
                   'close': v2['dlyprc'].abs(), 'ret': v2['dlyret'],
                   'volume': v2['dlyvol'], 'fuente': 'crsp_v2'})
print(f'bloque 2025 dsf_v2: {len(v2):,} filas, {v2.ticker.nunique()} tickers')

# --- Yahoo: respaldo 2025 (tickers sin dsf_v2) y cola 2026 ---
yh = pd.read_csv(EV / 'precios_yahoo_2025_2026.csv',
                 keep_default_na=False, na_values=[''], parse_dates=['date'])
yh = yh.sort_values(['ticker', 'date'])
yh['ret'] = yh.groupby('ticker')['close'].pct_change()   # incluye nov-dic 2024 para el arranque
yh26 = yh[yh.date >= '2026-01-01'].copy()
yh26['fuente'] = 'yahoo'
yh25 = yh[(yh.date >= '2025-01-01') & (yh.date <= '2025-12-31') &
          ~yh.ticker.isin(set(v2.ticker))].copy()
yh25['fuente'] = 'yahoo'
print(f'respaldo yahoo 2025: {yh25.ticker.nunique()} tickers fuera del universo CRSP')

cols = ['ticker', 'date', 'close', 'ret', 'volume', 'fuente']
panel26 = pd.concat([crsp[cols], v2[cols], yh25[cols], yh26[cols]],
                    ignore_index=True).sort_values(['ticker', 'date'])
panel26 = panel26.drop_duplicates(['ticker', 'date'])
panel26.to_csv(EV / 'panel_precios_2020_2026.csv', index=False)
print(f'panel 2020-2026: {len(panel26):,} renglones | tickers: {panel26.ticker.nunique()}')
print(panel26.groupby('fuente').size().to_string())
print('\nGME empalme dic-2025 / ene-2026:')
print(panel26[(panel26.ticker == 'GME') &
              (panel26.date.between('2025-12-29', '2026-01-05'))].to_string(index=False))


panel 2020-2026: 1,023,149 renglones | tickers: 1012
fuente
crsp     935089
yahoo     88060

GME empalme dic-2024 / ene-2025:
ticker       date  close       ret     volume fuente
   GME 2024-12-27  32.20 -0.023947 10141662.0   crsp
   GME 2024-12-30  32.01 -0.005901  9593140.0   crsp
   GME 2024-12-31  31.34 -0.020931  7395297.0   crsp
   GME 2025-01-02  30.66 -0.021698  7979700.0  yahoo
   GME 2025-01-03  31.65  0.032290  7461800.0  yahoo
   GME 2025-01-06  32.82  0.036967 12609400.0  yahoo


In [4]:
# Celda X4 - Cruce eventos-precios: retornos, volumen anormal y desfase atencion-precio
por_ticker = {t: g.set_index('date').sort_index()
              for t, g in panel26.groupby('ticker')}

filas = []
for _, e in cat.iterrows():
    g = por_ticker.get(e.ticker)
    if g is None:
        continue
    ini, pico, fin = e.fecha_inicio, e.fecha_pico, e.fecha_fin
    pre = g.loc[ini - pd.Timedelta(days=30): ini - pd.Timedelta(days=1)]
    evento = g.loc[ini: fin]
    ventana = g.loc[ini - pd.Timedelta(days=5): fin + pd.Timedelta(days=5)]
    if len(evento) < 2 or len(pre) < 5 or len(ventana) < 2:
        continue

    p_ini = pre['close'].iloc[-1]                    # ultimo cierre antes del encendido
    p_pico = evento['close'].asof(pico)
    p_fin = evento['close'].iloc[-1]
    base_vol = pre['volume'].mean()

    f_pico_px = ventana['close'].idxmax()
    f_pico_vol = ventana['volume'].idxmax()

    filas.append({
        'ticker': e.ticker,
        'fecha_inicio': ini.date(), 'fecha_pico': pico.date(), 'fecha_fin': fin.date(),
        'duracion_dias': e.duracion_dias, 'menciones_evento': e.menciones_evento,
        'amplitud': e.amplitud, 'regimen_lento': e.regimen_lento,
        'ret_encendido_pico': round(p_pico / p_ini - 1, 4) if p_ini and pd.notna(p_pico) else np.nan,
        'ret_pico_fin': round(p_fin / p_pico - 1, 4) if pd.notna(p_pico) and p_pico else np.nan,
        'ret_max_evento': round(ventana['close'].max() / p_ini - 1, 4) if p_ini else np.nan,
        'vol_ratio_evento': round(evento['volume'].mean() / base_vol, 2) if base_vol else np.nan,
        'desfase_precio_dias': (f_pico_px - pico).days,
        'desfase_volumen_dias': (f_pico_vol - pico).days,
    })

cx = pd.DataFrame(filas)
cx.to_csv(EV / 'eventos_con_precios.csv', index=False)

print(f'eventos cruzados con precios: {len(cx)} de {len(cat)}')
print(f"\nretorno encendido a pico de atencion: mediana {cx.ret_encendido_pico.median():+.1%} | "
      f"p75 {cx.ret_encendido_pico.quantile(.75):+.1%}")
print(f"retorno pico a fin (el desinfle):     mediana {cx.ret_pico_fin.median():+.1%} | "
      f"p25 {cx.ret_pico_fin.quantile(.25):+.1%}")
print(f"volumen evento / base pre-evento:     mediana {cx.vol_ratio_evento.median():.1f}x | "
      f"p75 {cx.vol_ratio_evento.quantile(.75):.1f}x")

print('\ndesfase pico de PRECIO vs pico de atencion (dias; negativo = precio primero):')
print(cx.desfase_precio_dias.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())
print(f"precio y atencion pican el MISMO dia: {(cx.desfase_precio_dias == 0).mean():.1%}")
print(f"a lo mas 1 dia de distancia:          {(cx.desfase_precio_dias.abs() <= 1).mean():.1%}")

print('\ndesfase pico de VOLUMEN vs pico de atencion (dias):')
print(f"mismo dia: {(cx.desfase_volumen_dias == 0).mean():.1%} | "
      f"a lo mas 1 dia: {(cx.desfase_volumen_dias.abs() <= 1).mean():.1%}")

print('\nGME enero 2021 como verificacion:')
print(cx[(cx.ticker == 'GME') & (cx.fecha_inicio == pd.Timestamp('2021-01-13').date())]
      .to_string(index=False))

eventos cruzados con precios: 2675 de 2791

retorno encendido a pico de atencion: mediana +4.6% | p75 +24.5%
retorno pico a fin (el desinfle):     mediana -0.3% | p25 -7.4%
volumen evento / base pre-evento:     mediana 1.9x | p75 3.7x

desfase pico de PRECIO vs pico de atencion (dias; negativo = precio primero):
count    2675.0
mean        1.0
std        11.1
min      -133.0
10%        -8.0
25%        -3.0
50%         0.0
75%         6.0
90%        11.0
max       126.0
precio y atencion pican el MISMO dia: 17.8%
a lo mas 1 dia de distancia:          30.4%

desfase pico de VOLUMEN vs pico de atencion (dias):
mismo dia: 35.1% | a lo mas 1 dia: 56.0%

GME enero 2021 como verificacion:
ticker fecha_inicio fecha_pico  fecha_fin  duracion_dias  menciones_evento  amplitud  regimen_lento  ret_encendido_pico  ret_pico_fin  ret_max_evento  vol_ratio_evento  desfase_precio_dias  desfase_volumen_dias
   GME   2021-01-13 2021-01-28 2021-02-04             23            865463     124.2          Fals